In [ ]:
import pandas as pd, zipfile, glob
z = glob.glob('/home/ibm-user/work/*.zip')[0]
zipfile.ZipFile(z).extractall('/home/ibm-user/work/data')
f = glob.glob('/home/ibm-user/work/data/*.csv')[0]
df = pd.read_csv(f)
print(df.shape)
print(df.isFraud.mean())
print(df.columns.tolist())
print(df.head())
print(df.groupby('type').isFraud.sum())

In [ ]:
import importlib
for m in ['cryptography','sklearn','xgboost','lightgbm']:
    try: print(m, importlib.import_module(m).__version__)
    except Exception as e: print(m, 'MISSING', e)

In [ ]:
%pip install pycryptodome

In [ ]:
from Crypto.Cipher import AES
print('AES-GCM available')

In [ ]:
import os, hmac, hashlib, secrets, json
from Crypto.Cipher import AES

V = '/home/ibm-user/work/vault'; os.makedirs(V, exist_ok=True)
def load_key(name):
    p = f'{V}/{name}.key'
    if not os.path.exists(p):
        with open(p,'wb') as fh: fh.write(secrets.token_bytes(32))
        os.chmod(p, 0o600)
    return open(p,'rb').read()
HMAC_KEY, AES_KEY = load_key('hmac'), load_key('aes')

def tokenize(x):  # same ID -> same token, so history features still work
    return 'TK_' + hmac.new(HMAC_KEY, x.encode(), hashlib.sha256).hexdigest()[:16]
def encrypt(x):
    n = secrets.token_bytes(12)
    ct, tag = AES.new(AES_KEY, AES.MODE_GCM, nonce=n).encrypt_and_digest(x.encode())
    return (n + tag + ct).hex()
def decrypt(h):
    b = bytes.fromhex(h)
    return AES.new(AES_KEY, AES.MODE_GCM, nonce=b[:12]).decrypt_and_verify(b[28:], b[12:28]).decode()

d = df[df.type.isin(['TRANSFER','CASH_OUT'])].sort_values('step').head(200_000).copy()
ids = pd.unique(pd.concat([d.nameOrig, d.nameDest]))
vault = {tokenize(i): encrypt(i) for i in ids}
d['token_customer'] = d.nameOrig.map(tokenize)
d['token_counterparty'] = d.nameDest.map(tokenize)

safe = d[['step','type','amount','token_customer','token_counterparty','isFraud']]
assert not safe.astype(str).apply(lambda c: c.str.contains(r'^[CM]\d{6,}', regex=True)).any().any()
t = safe.token_customer.iloc[0]
assert decrypt(vault[t]) == d.nameOrig.iloc[0]
safe.to_csv('/home/ibm-user/work/data/tokenized_sample.csv', index=False)
json.dump(vault, open(f'{V}/vault.json','w'))
print(len(ids), 'IDs tokenized;', safe.isFraud.mean(), 'fraud rate')
print(safe.head(3))

In [ ]:
d = df[df.type.isin(['TRANSFER','CASH_OUT'])].copy()
print('rows', len(d), '| unique senders', d.nameOrig.nunique(), '| unique receivers', d.nameDest.nunique())
fr = d[d.isFraud==1]
print('fraud rows', len(fr), '| unique receivers among fraud', fr.nameDest.nunique())
vc = d.nameDest.value_counts()
print('share of receivers seen more than once:', (vc>1).mean())
print(vc.head(3))

In [ ]:
d = df[df.type.isin(['TRANSFER','CASH_OUT'])].sort_values('step').copy()
for col, new in [('nameOrig','token_customer'), ('nameDest','token_counterparty')]:
    m = {i: tokenize(i) for i in d[col].unique()}
    d[new] = d[col].map(m)
full = d[['step','type','amount','token_customer','token_counterparty','isFraud']]
full.to_csv('/home/ibm-user/work/data/tokenized_full.csv', index=False)
print(full.shape, full.isFraud.sum())

In [ ]:
import numpy as np, xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

f = pd.read_csv('/home/ibm-user/work/data/tokenized_full.csv')
f = f.sort_values('step', kind='stable').reset_index(drop=True)
g = f.groupby('token_counterparty')
f['recv_prev_count'] = g.cumcount()
f['recv_prev_sum'] = g.amount.cumsum() - f.amount
f['amt_vs_recv_avg'] = f.amount / (f.recv_prev_sum / f.recv_prev_count.clip(lower=1)).clip(lower=1)
f['log_amount'] = np.log1p(f.amount)
f['is_transfer'] = (f.type == 'TRANSFER').astype(int)
f['hour'] = f.step % 24
feats = ['log_amount','is_transfer','hour','recv_prev_count','recv_prev_sum','amt_vs_recv_avg']

tr, te = f[f.step <= 600], f[f.step > 600]
print('train', len(tr), tr.isFraud.sum(), '| test', len(te), te.isFraud.sum())
m = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                      tree_method='hist', scale_pos_weight=50, n_jobs=3)
m.fit(tr[feats], tr.isFraud)
p = m.predict_proba(te[feats])[:, 1]
print('PR-AUC', average_precision_score(te.isFraud, p), '| ROC-AUC', roc_auc_score(te.isFraud, p))

In [ ]:
# 1) Proof: features from RAW IDs are identical to features from TOKENS
r = d[['step','type','amount','nameOrig','nameDest','isFraud']].reset_index(drop=True)
gr = r.groupby('nameDest')
r['recv_prev_count'] = gr.cumcount()
r['recv_prev_sum'] = gr.amount.cumsum() - r.amount
r['amt_vs_recv_avg'] = r.amount / (r.recv_prev_sum / r.recv_prev_count.clip(lower=1)).clip(lower=1)
r['log_amount'] = np.log1p(r.amount); r['is_transfer'] = (r.type=='TRANSFER').astype(int); r['hour'] = r.step % 24
print('same row order:', (r.step.values==f.step.values).all() and (r.amount.values==f.amount.values).all())
print('features identical:', np.allclose(r[feats].values, f[feats].values))

# 2) Alert-budget view on the test set
te = te.assign(p=p)
for k in [100, 300, 600, 1000, 2000]:
    top = te.nlargest(k, 'p')
    print(k, 'flagged | precision', round(top.isFraud.mean(),3), '| recall', round(top.isFraud.sum()/te.isFraud.sum(),3))

# 3) what the model relies on
print(dict(zip(feats, m.feature_importances_.round(3))))

In [ ]:
import os, json
EX = '/home/ibm-user/work/export'; os.makedirs(EX, exist_ok=True)
m.save_model(f'{EX}/model.json')

# receiver history at the end of the data (the live API starts from this)
st = f.groupby('token_counterparty').amount.agg(['count','sum'])
json.dump({k:[int(c), float(s)] for k,c,s in zip(st.index, st['count'], st['sum'])}, open(f'{EX}/state.json','w'))

REASONS = {'log_amount':'Unusually large amount', 'is_transfer':'Transaction type is TRANSFER',
           'hour':'Unusual time of day', 'recv_prev_count':'Receiver account is new or rare',
           'recv_prev_sum':'Receiver inflow history', 'amt_vs_recv_avg':'Amount unusual for this receiver'}

top = te.nlargest(1000, 'p').copy()
contrib = m.get_booster().predict(xgb.DMatrix(top[feats]), pred_contribs=True)[:, :-1]
top['top_reason'] = [REASONS[feats[i]] for i in contrib.argmax(axis=1)]
top['risk_score'] = top.p.round(4)
top['tx_id'] = ['TX%07d' % i for i in top.index]
top['timestamp'] = (pd.Timestamp('2026-10-01') + pd.to_timedelta(top.step, unit='h')).astype(str)
alerts = top[['tx_id','token_customer','token_counterparty','amount','type','risk_score','top_reason','timestamp']]
alerts = alerts.sort_values('risk_score', ascending=False)
assert not alerts.astype(str).apply(lambda c: c.str.contains(r'^[CM]\d{6,}', regex=True)).any().any()
alerts.to_csv(f'{EX}/alerts.csv', index=False)

# vault ONLY for the IDs that appear in alerts (reviewer re-identification)
rows = r.loc[top.index]
vault_alerts = {tokenize(x): encrypt(x) for x in pd.unique(pd.concat([rows.nameOrig, rows.nameDest]))}
json.dump(vault_alerts, open(f'{EX}/vault_alerts.json','w'))
print(alerts.shape, '| precision of these alerts:', round(top.isFraud.mean(), 3))
print(alerts.head(3).to_string())